# HWK-Aali (آلي) — Colab Quickstart

تجربة تدريب سريعة (smoke test) لنموذج HWK-Aali على GPU مجاني في Colab — منفصلة عن تدريب المرحلة الرئيسية (Phase A) على بطاقة الشاشة المحلية.

**الغرض:** التأكّد أن الكود يعمل ويدرّب فعلياً على GPU حقيقي—ليس لإنتاج نموذج نهائي. هذا تدريب SFT صغير على `data/agent_instructions.jsonl` باستخدام مرمّز أحرف (byte tokenizer) ونموذج صغير جداً، مصمّم لينتهي خلال دقائق على T4 المجانية.

**الخطوات:**
1. `Runtime -> Change runtime type -> T4 GPU` (مجاني)
2. شغّل الخلايا بالترتيب
3. عند الوصول لخلية "رفع الحزمة" سيطلب منك رفع ملف `hwk_colab_bundle.zip` (مرفق مع هذا الدفتر)

In [ ]:
# 1) فحص GPU
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

In [ ]:
# 2) رفع الحزمة الجاهزة hwk_colab_bundle.zip
from google.colab import files
import os

if not os.path.exists("hwk_colab_bundle.zip"):
    print("ارفع ملف hwk_colab_bundle.zip الآن:")
    uploaded = files.upload()
else:
    print("الملف موجود بالفعل.")

In [ ]:
# 3) فك الضغط والدخول للمجلد
!unzip -o -q hwk_colab_bundle.zip
%cd colab_bundle
!ls -la

In [ ]:
# 4) المكتبات — torch وnumpy موجودان في Colab عادةً； sentencepiece فقط إذا أردت استخدام مرمّز BPE لاحقاً
!pip -q install sentencepiece

In [ ]:
# 5) تدريب سريع صغير جداً (smoke test) — بضع تجريبيو، بضع دقائق على T4
!python train_scratch.py \
  --data data/agent_instructions.jsonl \
  --output-dir /content/hwk-smoke-out \
  --mirror-dir '' \
  --context 256 --d-model 256 --heads 8 --layers 6 \
  --batch-size 8 --gradient-accumulation 2 \
  --max-steps 400 --save-steps 200 --eval-steps 100 --log-steps 20 \
  --warmup-steps 50 --dtype fp16 \
  --device auto

In [ ]:
# 6) تأكّد من وجود النقطة المحفوظة وحمّلها لجهازك إن أردت
!ls -la /content/hwk-smoke-out
from google.colab import files as _f
# _f.download('/content/hwk-smoke-out/checkpoint.pt')  # ألغِ التعليق للتحميل

## ما التالي؟

هذا الدفتر للتجربة فقط (نموذج صغير + بيانات قليلة). التدريب الحقيقي (Phase A) على بيانات الـ 139GB المرمزة يجب أن يتم على جهازك (بطاقة الشاشة المحلية) عبر `scripts/resume_training.bat` — راجع `docs/training.md` في المستودع للتفاصيل الكاملة.